# Section 6: Deployment Notes

## How This Model Is Deployed

### Overview

The donor lapse risk classifier is deployed as a backend scoring service integrated
into the organization's web application. It does **not** run on a fixed nightly schedule.
Instead it uses a two-mode deployment strategy designed around the reality that donors
give on average twice per year — meaning day-to-day score changes are rarely meaningful,
but threshold crossings are.

---

### Mode 1: Monthly Batch Re-Score (Strategic Layer)

**When it runs:** First day of each month, triggered by a scheduled job (cron).

**What it does:**
- Loads all `Active` supporters from the database
- Re-engineers all features from live donation history
- Scores every supporter and writes results to a `donor_risk_scores` table
- Updates the Admin Dashboard watchlist — top donors ranked by `risk_score × lifetime_value`
- Generates the strategic insight card: aggregate patterns by acquisition channel,
  supporter type, and campaign participation (e.g., "5 of 7 high-risk donors were
  acquired via Word of Mouth")

**Why monthly, not nightly:** Donor behavior changes on a timescale of weeks and months,
not hours. Running nightly would surface the same names repeatedly, causing alert fatigue.
A monthly cadence matches the executive director's natural review rhythm and ensures
the watchlist feels fresh and actionable each time it is consulted.

---

### Mode 2: Personal Threshold Trigger (Operational Layer)

**When it runs:** Continuously — evaluated each time a donation event is recorded
or on a weekly lightweight check.

**What it fires on:** A donor is added to the active watchlist mid-cycle only when
they cross a personal threshold, defined as:

```
days_since_last_donation > (personal_avg_gap × 1.5)
```

This means Rosa, who normally gives every 38 days, gets flagged at day 57 —
not because a calendar flipped, but because her own behavior has shifted.
A donor who normally gives once a year is not flagged at day 180.

**Why personal thresholds:** Raw recency penalizes infrequent donors unfairly.
Relative recency captures genuine behavioral drift from each donor's own baseline.

---

### The Deployment Contract: What the Model Must Return

Every scored donor produces a structured output object consumed by the frontend.
The API endpoint (`GET /api/donors/{id}/risk`) returns:

```json
{
  "supporter_id": 12,
  "risk_tier": "High",
  "risk_score": 0.847,
  "days_since_last_donation": 147,
  "personal_avg_gap_days": 38,
  "gap_ratio": 3.87,
  "lifetime_value_php": 4200,
  "priority_score": 3567,
  "threshold_crossed": true,
  "risk_reasons": [
    {
      "code": "GAP_OVERDUE",
      "label": "147 days since last gift — 3.9x their usual gap",
      "direction": "increases_risk"
    },
    {
      "code": "AMOUNT_DECLINING",
      "label": "Recent gifts declining — avg dropped from ₱850 to ₱490",
      "direction": "increases_risk"
    },
    {
      "code": "CHANNEL_HIGH_RISK",
      "label": "Word of Mouth channel — historically 64% lapse rate",
      "direction": "increases_risk"
    },
    {
      "code": "CAMPAIGN_ENGAGED",
      "label": "Has donated during 2 named campaigns",
      "direction": "decreases_risk"
    }
  ],
  "suggested_action": "Personal outreach within 7 days. Reference Year-End Hope campaign history.",
  "last_scored_at": "2024-12-01T00:00:00Z",
  "snooze_until": null
}
```

**Note on `risk_score`:** The raw probability (0.847) is stored in the database for
ranking purposes but is never shown to staff. The frontend renders only `risk_tier`
(Low / Medium / High) and the plain-language `risk_reasons` array.

**Note on `priority_score`:** Calculated as `risk_score × lifetime_value_php`. This
ensures a high-value donor showing medium risk outranks a low-value donor showing
high risk on the watchlist. The executive director's limited time is directed at
the donors whose lapse would hurt most.

---

### Human-in-the-Loop: The Snooze Mechanism

Once a staff member takes outreach action on a flagged donor, they mark the donor
as contacted via a button on the donor profile page. This writes a `contacted_at`
timestamp and a `snooze_until` date (default: 45 days) to the `donor_risk_scores`
table. The donor is removed from the active watchlist for the snooze period
regardless of model score.

This prevents the core failure mode of automated watchlists: surfacing the same
name every day after action has already been taken, causing staff to stop trusting
the list.

```
Snooze states:
- null          → donor is eligible for watchlist based on score
- future date   → donor is snoozed, hidden from watchlist
- past date     → snooze expired, model re-evaluates for watchlist inclusion
```

The snooze is a human override — not a model output. The model continues scoring
the donor in the background; only the watchlist display is suppressed.

---

### Where Integration Code Lives

| Component | Location in Repo |
|---|---|
| Scoring script (batch) | `ml-pipelines/donor-lapse-risk/score.py` |
| Feature engineering pipeline | `ml-pipelines/donor-lapse-risk/features.py` |
| Trained model artifact | `ml-pipelines/donor-lapse-risk/model.pkl` |
| API endpoint | `backend/Controllers/DonorRiskController.cs` |
| Watchlist component | `frontend/src/components/DonorWatchlist.tsx` |
| Donor risk badge | `frontend/src/components/DonorRiskBadge.tsx` |
| Snooze handler | `backend/Controllers/DonorRiskController.cs` → `POST /api/donors/{id}/risk/snooze` |
| Scheduled batch job | `backend/Jobs/MonthlyDonorReScoringJob.cs` |

---

### Deployment Architecture Summary

```
Monthly cron job
    → features.py (rebuild features from live DB)
    → model.pkl (score all active supporters)
    → donor_risk_scores table (write results)
    → Admin Dashboard (read top N by priority_score)

Donor profile page
    → GET /api/donors/{id}/risk
    → renders risk tier badge + reason cards + suggested action

Staff marks donor as contacted
    → POST /api/donors/{id}/risk/snooze
    → donor removed from watchlist for 45 days

Threshold trigger (weekly lightweight check)
    → queries donors where gap_ratio > 1.5 and snooze_until is null
    → adds mid-cycle flags to watchlist without waiting for monthly batch
```

---

### What This Looks Like for the Executive Director

The executive director never interacts with the model directly. Their experience is:

1. **Monthly:** Opens the dashboard, sees a refreshed watchlist of 5–10 donors
   ranked by priority, each with a plain-language reason and a suggested action.

2. **Mid-month:** If a donor crosses their personal threshold between monthly runs,
   they appear on the watchlist with a "New flag" indicator.

3. **After outreach:** Staff clicks "Mark as contacted" on the donor profile.
   The donor disappears from the watchlist for 45 days. If they donate during
   that window, the risk score resets automatically on the next monthly run.

4. **Quarterly:** The Reports page shows aggregate model findings —
   which acquisition channels produce the most loyal donors, which campaign
   types correlate with repeat giving — informing the org's strategic decisions
   without requiring any ML literacy from staff.